# 피부 병변 세그멘테이션 — ISIC 2018 불균형 손실 함수 비교

## [project-planner] 연구 계획

| 단계 | 내용 | 완료 기준 |
|------|------|-----------|
| 1 | 환경 설정 + 데이터 다운로드 | `len(matched_imgs) > 0` |
| 2 | 클래스 비율 계산 | `class_counts = [bg, fg]` 출력 |
| 3 | U-Net 모델 정의 | `build_unet()` 호출 성공 |
| 4 | Optuna alpha 탐색 | `best_alpha_plwce`, `best_alpha_pwce` 확보 |
| 5 | 5종 Loss 비교 학습 | `all_results` 딕셔너리 완성 |
| 6 | 시각화 + 평가 | PNG 저장, JSON 저장 |

**도메인**: ISIC 2018 Task 1 — Skin Lesion Segmentation  
**모델**: U-Net (ResNet34 백본, ImageNet 사전학습)  
**불균형**: BG vs 병변 (병변 크기 편차 큼)  
**비교 Loss**: `ce_dice`, `wce_dice`, `lwce_dice`, `plwce_dice`, `cb_dice`  
**평가 지표**: Dice, IoU, Sensitivity, Specificity, AUC

---

### 리스크 및 대응
| 리스크 | 대응 |
|--------|------|
| kaggle 데이터 다운로드 실패 | 수동 배치 경로 `DATA_DIR` 수정 |
| GPU 메모리 부족 | `BATCH_SIZE` 줄이기 (기본 8 → 4) |
| optuna 미설치 | Cell 1에서 자동 설치 |

In [1]:
# ── Cell 0: 환경 설정 + 패키지 설치 ──────────────────────────────────────────
import subprocess, sys

for pkg in ['segmentation-models-pytorch', 'optuna']:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

import os, warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import cv2
import glob
import random
import json
from tqdm import tqdm

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, confusion_matrix

import segmentation_models_pytorch as smp
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

import kagglehub

# custom_losses.py — 올바른 경로 (imbalanced, inbalanced 아님)
sys.path.insert(0, '/root/imbalanced-data-LWCE/medical_data')
from custom_losses import get_loss_function

# ── 실험 설정 ─────────────────────────────────────────────────────────────────
IMG_SIZE    = 256   # 입력 해상도
BATCH_SIZE  = 8    # GPU 메모리 부족 시 4로 줄이기
NUM_WORKERS = 2
SEED        = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print('환경 설정 완료')

Device: cuda
환경 설정 완료


In [ ]:
# ── Cell 1: 데이터셋 다운로드 + Dataset 클래스 + DataLoader ──────────────────

# 1-1. ISIC 2018 Task 1 다운로드 (kagglehub)
# 데이터셋 구조:
#   ISIC2018_Task1-2_Training_Input/     → 이미지 (.jpg)
#   ISIC2018_Task1_Training_GroundTruth/ → 마스크 (_segmentation.png)
print('ISIC 2018 데이터셋 다운로드 중...')
path = kagglehub.dataset_download('tschandl/isic2018-challenge-task1-data-segmentation')
print(f'Path to dataset files: {path}')

IMG_DIR  = os.path.join(path, 'ISIC2018_Task1-2_Training_Input')
MASK_DIR = os.path.join(path, 'ISIC2018_Task1_Training_GroundTruth')

# 1-2. 이미지 / 마스크 경로 로드 (확정된 폴더 구조 사용)
all_images = sorted(glob.glob(os.path.join(IMG_DIR,  '*.jpg')))
all_masks  = sorted(glob.glob(os.path.join(MASK_DIR, '*_segmentation.png')))

# ID 기반 매칭 (이미지: ISIC_xxxxxxx  ↔  마스크: ISIC_xxxxxxx_segmentation)
img_id  = {os.path.splitext(os.path.basename(p))[0]: p for p in all_images}
mask_id = {
    os.path.splitext(os.path.basename(p))[0].replace('_segmentation', ''): p
    for p in all_masks
}

common        = sorted(set(img_id.keys()) & set(mask_id.keys()))
matched_imgs  = [img_id[k]  for k in common]
matched_masks = [mask_id[k] for k in common]

assert len(common) > 0, 'ISIC 이미지-마스크 쌍을 찾을 수 없습니다.'
print(f'매칭된 이미지-마스크 쌍: {len(common)}')

# 1-3. Train / Val 분할 (8:2)
tr_imgs, val_imgs, tr_masks, val_masks = train_test_split(
    matched_imgs, matched_masks, test_size=0.2, random_state=SEED
)
print(f'Train: {len(tr_imgs)}  |  Val: {len(val_imgs)}')

# 1-4. Dataset 클래스
class ISICDataset(Dataset):
    """ISIC 2018 Task 1 Skin Lesion Segmentation Dataset.

    Args:
        img_paths:  이미지 경로 리스트
        mask_paths: 마스크 경로 리스트
        transform:  albumentations 변환 (None 가능)
    """

    def __init__(self, img_paths: list, mask_paths: list,
                 transform=None) -> None:
        self.img_paths  = img_paths
        self.mask_paths = mask_paths
        self.transform  = transform

    def __len__(self) -> int:
        return len(self.img_paths)

    def __getitem__(self, idx: int):
        img  = cv2.cvtColor(cv2.imread(self.img_paths[idx]),  cv2.COLOR_BGR2RGB)
        mask = cv2.imread(self.mask_paths[idx], cv2.IMREAD_GRAYSCALE)
        # 이진화: 흰색 영역(병변) = 1, 배경 = 0
        _, mask = cv2.threshold(mask, 127, 1, cv2.THRESH_BINARY)
        if self.transform:
            aug  = self.transform(image=img, mask=mask)
            img, mask = aug['image'], aug['mask']
        return img, mask.long()

# 1-5. Augmentation 설정
train_tf = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.2, rotate_limit=30, p=0.5),
    A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, p=0.4),
    A.Normalize(), ToTensorV2()
])
val_tf = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(), ToTensorV2()
])

train_loader = DataLoader(
    ISICDataset(tr_imgs,  tr_masks,  train_tf),
    batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS, pin_memory=True
)
val_loader = DataLoader(
    ISICDataset(val_imgs, val_masks, val_tf),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True
)
print('DataLoader 구성 완료')

In [ ]:
# ── Cell 2: 클래스 비율 계산 (rules.md 규칙: 픽셀 단위, 학습 데이터만) ────────
print('클래스 비율 계산 중 (학습 마스크 전체)...')

bg_pixels, fg_pixels = 0, 0
for mp in tqdm(tr_masks, desc='Counting pixels'):
    m = cv2.imread(mp, cv2.IMREAD_GRAYSCALE)
    fg_pixels += int((m > 127).sum())
    bg_pixels += int((m <= 127).sum())

class_counts = [bg_pixels, fg_pixels]

print(f'\nBG (배경):       {bg_pixels:>15,} pixels')
print(f'FG (병변):       {fg_pixels:>15,} pixels')
print(f'불균형 비율:     BG:FG = {bg_pixels / fg_pixels:.1f}:1')
print(f'\ncclass_counts = {class_counts}')

In [ ]:
# ── Cell 3: U-Net 모델 + 유틸리티 함수 ──────────────────────────────────────

def build_unet() -> nn.Module:
    """ResNet34 백본 U-Net (ImageNet 사전학습, 1채널 출력)."""
    return smp.Unet(
        encoder_name='resnet34',
        encoder_weights='imagenet',
        in_channels=3,
        classes=1,
        activation=None,   # 로짓 출력 — sigmoid는 평가 시 적용
    ).to(device)


def to_2ch_logits(p: torch.Tensor) -> torch.Tensor:
    """1채널 sigmoid 출력을 2채널 대칭 logit으로 변환.

    [-p, p] 방식 사용 → sigmoid(p)와 수학적으로 동일.
    [zeros, p] 방식은 배경 logit 0 고정으로 학습 불안정 → 금지.
    """
    return torch.cat([-p, p], dim=1)


def compute_val_metrics(model: nn.Module, loader: DataLoader) -> dict:
    """검증 세트에서 Dice / Sensitivity / Specificity / AUC 계산.

    Returns:
        dict: {'Dice': float, 'Sens': float, 'Spec': float, 'AUC': float}
    """
    model.eval()
    dices, sens_list, spec_list, aucs = [], [], [], []

    with torch.no_grad():
        for imgs, masks in loader:
            imgs, masks = imgs.to(device), masks.to(device)
            logits = model(imgs)                       # (B, 1, H, W)
            prob   = torch.sigmoid(logits).squeeze(1)  # (B, H, W)
            pred   = (prob > 0.5).long()               # (B, H, W)

            for b in range(imgs.size(0)):
                p_flat = pred[b].cpu().numpy().flatten()
                t_flat = masks[b].cpu().numpy().flatten()
                r_flat = prob[b].cpu().numpy().flatten()

                # AUC
                try:
                    aucs.append(roc_auc_score(t_flat, r_flat))
                except ValueError:
                    aucs.append(0.5)

                # Dice / Sens / Spec
                tn, fp, fn, tp = confusion_matrix(
                    t_flat, p_flat, labels=[0, 1]
                ).ravel()
                dices.append((2. * tp) / (2. * tp + fp + fn + 1e-8))
                sens_list.append(tp / (tp + fn + 1e-8))
                spec_list.append(tn / (tn + fp + 1e-8))

    return {
        'Dice': float(np.mean(dices)),
        'Sens': float(np.mean(sens_list)),
        'Spec': float(np.mean(spec_list)),
        'AUC':  float(np.mean(aucs)),
    }


def compute_val_dice(model: nn.Module, loader: DataLoader) -> float:
    """빠른 Val Dice 계산 (Optuna proxy 및 학습 루프용)."""
    model.eval()
    dice_sum = 0.0
    with torch.no_grad():
        for imgs, masks in loader:
            imgs, masks = imgs.to(device), masks.to(device)
            prob  = torch.sigmoid(model(imgs)).squeeze(1)
            pred  = (prob > 0.5).long()
            inter = (pred.float() * masks.float()).sum()
            union = pred.float().sum() + masks.float().sum()
            dice_sum += (2. * inter / (union + 1e-8)).item() if union > 0 else 1.0
    return dice_sum / len(loader)


# 모델 파라미터 수 확인
test_model = build_unet()
n_params   = sum(p.numel() for p in test_model.parameters() if p.requires_grad)
print(f'U-Net (ResNet34) 파라미터 수: {n_params:,}')
del test_model
print('모델 + 유틸리티 함수 준비 완료')

In [ ]:
# ── Cell 4: 학습 함수 ─────────────────────────────────────────────────────────

def train_unet(
    loss_name: str,
    alpha: float = 1.0,
    epochs: int = 30,
    lr: float = 1e-4,
    subset_ratio: float = 1.0,
    tag: str = '',
):
    """U-Net 학습 함수.

    Args:
        loss_name:    손실 함수 이름 (예: 'lwce_dice', 'plwce_dice')
        alpha:        PLWCE / PWCE 강도 파라미터
        epochs:       학습 에폭 수
        lr:           학습률
        subset_ratio: Optuna proxy용 데이터 축소 비율 (1.0 = 전체)
        tag:          저장 파일 구분 태그

    Returns:
        Tuple[nn.Module, dict, float]: (최고 모델, 히스토리, 최고 Val Dice)
    """
    model     = build_unet()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    # get_loss_function: 가중치 모드 + Loss 종류 통합 (rules.md 규칙)
    criterion = get_loss_function(loss_name, class_counts=class_counts, alpha=alpha)

    name = f'{loss_name}_alpha{alpha:.2f}' if alpha != 1.0 else loss_name
    if tag:
        name = f'{tag}_{name}'

    print(f"\n{'='*55}\nU-Net + {name}  (epochs={epochs})\n{'='*55}")

    # Optuna proxy용 데이터 축소
    if subset_ratio < 1.0:
        n = max(1, int(len(train_loader.dataset) * subset_ratio))
        sub_ds = torch.utils.data.Subset(
            train_loader.dataset,
            random.sample(range(len(train_loader.dataset)), n)
        )
        loader = DataLoader(sub_ds, batch_size=BATCH_SIZE,
                            shuffle=True, num_workers=NUM_WORKERS)
    else:
        loader = train_loader

    history   = {'loss': [], 'val_dice': []}
    best_dice = 0.0
    save_path = f'/tmp/best_unet_{name}.pth'

    for epoch in range(epochs):
        model.train()
        epoch_loss = 0.0

        for imgs, masks in tqdm(loader, desc=f'Ep{epoch+1:02d}/{epochs}', leave=False):
            imgs, masks = imgs.to(device), masks.to(device)
            optimizer.zero_grad()

            logits = model(imgs)                      # (B, 1, H, W)
            # to_2ch_logits: 1채널 → 2채널 대칭 변환 (핵심 패턴)
            loss   = criterion(to_2ch_logits(logits), masks)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()

        scheduler.step()
        avg_loss = epoch_loss / len(loader)
        val_dice = compute_val_dice(model, val_loader)

        history['loss'].append(avg_loss)
        history['val_dice'].append(val_dice)

        print(f'Ep{epoch+1:02d} | Loss: {avg_loss:.4f} | Val Dice: {val_dice:.4f}', end='')
        if val_dice > best_dice:
            best_dice = val_dice
            torch.save(model.state_dict(), save_path)
            print('  <- Best!', end='')
        print()

    model.load_state_dict(torch.load(save_path, weights_only=True))
    print(f'최고 Val Dice: {best_dice:.4f}')
    return model, history, best_dice


print('train_unet() 함수 준비 완료')

In [ ]:
# ── Cell 5: [선택] Optuna alpha 탐색 (plwce, pwce) ───────────────────────────
# tabular_data/scr/optuna_tuner_alpha_only.py 패턴 기반
# proxy 설정: subset_ratio=0.15, epochs=5, n_trials=20

ALPHA_LOW    = 2.5
ALPHA_HIGH   = 15.0
PROXY_EPOCHS = 5
PROXY_SUBSET = 0.15
N_TRIALS     = 20


def make_objective(loss_name: str):
    """loss_name 별 Optuna objective 생성."""
    def objective(trial: optuna.Trial) -> float:
        alpha = trial.suggest_float('alpha', ALPHA_LOW, ALPHA_HIGH)
        try:
            _, _, dice = train_unet(
                loss_name    = loss_name,
                alpha        = alpha,
                epochs       = PROXY_EPOCHS,
                subset_ratio = PROXY_SUBSET,
                tag          = f'trial{trial.number}',
            )
            return dice
        except Exception as e:
            print(f'Trial {trial.number} 실패: {e}')
            return 0.0
    return objective


# ── PLWCE alpha 탐색 ──────────────────────────────────────────────────────────
print(f'[Optuna] PLWCE alpha 탐색  (범위: {ALPHA_LOW}~{ALPHA_HIGH}, {N_TRIALS} trials)')
study_plwce = optuna.create_study(
    direction  = 'maximize',
    study_name = 'unet_isic_plwce_alpha',
    pruner     = optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=2),
)
study_plwce.optimize(make_objective('plwce_dice'), n_trials=N_TRIALS)
best_alpha_plwce = study_plwce.best_params['alpha']
print(f'[PLWCE] 최적 alpha = {best_alpha_plwce:.4f}  (Val Dice = {study_plwce.best_value:.4f})')

# ── PWCE alpha 탐색 ───────────────────────────────────────────────────────────
print(f'\n[Optuna] PWCE alpha 탐색  (범위: {ALPHA_LOW}~{ALPHA_HIGH}, {N_TRIALS} trials)')
study_pwce = optuna.create_study(
    direction  = 'maximize',
    study_name = 'unet_isic_pwce_alpha',
    pruner     = optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=2),
)
study_pwce.optimize(make_objective('pwce_dice'), n_trials=N_TRIALS)
best_alpha_pwce = study_pwce.best_params['alpha']
print(f'[PWCE]  최적 alpha = {best_alpha_pwce:.4f}  (Val Dice = {study_pwce.best_value:.4f})')

# ── Optuna 결과 저장 ──────────────────────────────────────────────────────────
optuna_results = {
    'plwce': {
        'best_alpha': best_alpha_plwce,
        'best_proxy_dice': study_plwce.best_value,
        'trials': [
            {'number': t.number, 'alpha': t.params.get('alpha'), 'value': t.value}
            for t in study_plwce.trials if t.value is not None
        ],
    },
    'pwce': {
        'best_alpha': best_alpha_pwce,
        'best_proxy_dice': study_pwce.best_value,
        'trials': [
            {'number': t.number, 'alpha': t.params.get('alpha'), 'value': t.value}
            for t in study_pwce.trials if t.value is not None
        ],
    },
}
with open('/tmp/isic_optuna_results.json', 'w') as f:
    json.dump(optuna_results, f, indent=2, ensure_ascii=False)
print('\nOptuna 결과 저장: /tmp/isic_optuna_results.json')

# ── 탐색 결과 시각화 ──────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, study, name in [
    (axes[0], study_plwce, 'PLWCE'),
    (axes[1], study_pwce,  'PWCE'),
]:
    trials  = [t for t in study.trials if t.value is not None]
    alphas  = [t.params['alpha'] for t in trials]
    values  = [t.value for t in trials]
    best_a  = study.best_params['alpha']
    best_v  = study.best_value

    ax.scatter(alphas, values, alpha=0.5, s=40, label='Trials')
    ax.axvline(best_a, color='red', linestyle='--', label=f'Best α={best_a:.2f}')
    ax.scatter([best_a], [best_v], color='red', s=100, zorder=5)
    ax.set_xlabel('alpha')
    ax.set_ylabel('Val Dice (proxy)')
    ax.set_title(f'{name} alpha 탐색 결과')
    ax.legend()
    ax.grid(True)

plt.tight_layout()
plt.savefig('/tmp/isic_optuna_alpha_search.png', dpi=100)
plt.show()
print('탐색 결과 저장: /tmp/isic_optuna_alpha_search.png')

In [ ]:
# ── Cell 6: 전체 Loss 비교 실험 ───────────────────────────────────────────────
# rules.md 규칙: ce_dice 기준선 + 최소 4종 이상 비교

FINAL_EPOCHS = 30
FINAL_LR     = 1e-4

# Optuna 탐색 결과 로드 (Cell 5 미실행 시 JSON fallback)
try:
    _ = best_alpha_plwce
except NameError:
    try:
        with open('/tmp/isic_optuna_results.json') as f:
            d = json.load(f)
        best_alpha_plwce = d['plwce']['best_alpha']
        best_alpha_pwce  = d['pwce']['best_alpha']
        print(f'Optuna 결과 로드: PLWCE alpha={best_alpha_plwce:.4f}, PWCE alpha={best_alpha_pwce:.4f}')
    except FileNotFoundError:
        # Cell 5를 건너뛴 경우 기본값 사용
        best_alpha_plwce = 5.0
        best_alpha_pwce  = 5.0
        print(f'Optuna 결과 없음 → 기본값 alpha=5.0 사용')

# ── 실험 목록 ─────────────────────────────────────────────────────────────────
experiments = [
    ('ce_dice',   1.0,              'CE+Dice        (기준선)'),
    ('wce_dice',  1.0,              'WCE+Dice'),
    ('lwce_dice', 1.0,              'LWCE+Dice'),
    ('plwce_dice', best_alpha_plwce, f'PLWCE+Dice     (alpha={best_alpha_plwce:.2f})'),
    ('cb_dice',   1.0,              'CB+Dice'),
]

all_results = {}
for loss_name, alpha, label in experiments:
    model, history, best_dice = train_unet(
        loss_name = loss_name,
        alpha     = alpha,
        epochs    = FINAL_EPOCHS,
        lr        = FINAL_LR,
        tag       = 'final',
    )
    all_results[label] = {
        'model':     model,
        'history':   history,
        'best_dice': best_dice,
        'loss_name': loss_name,
        'alpha':     alpha,
    }

# ── 1차 요약 출력 ─────────────────────────────────────────────────────────────
print('\n' + '='*45)
print('[Loss 비교 실험 1차 요약 — Val Dice]')
print(f"{'Loss':<35} {'Best Val Dice':>13}")
print('-' * 50)
for label, v in all_results.items():
    print(f"{label:<35} {v['best_dice']:>13.4f}")

In [ ]:
# ── Cell 7: 시각화 ── 학습 곡선 + 예측 결과 ──────────────────────────────────

COLORS = ['#4878D0', '#EE854A', '#6ACC65', '#D65F5F', '#B47CC7']

# 7-1. 학습 곡선 (Loss + Val Dice)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
for i, (label, v) in enumerate(all_results.items()):
    h = v['history']
    ax1.plot(h['loss'],     label=label, color=COLORS[i % len(COLORS)])
    ax2.plot(h['val_dice'], label=label, color=COLORS[i % len(COLORS)])

ax1.set_title('Train Loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.legend(fontsize=8)
ax1.grid(True)

ax2.set_title('Val Dice')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Dice')
ax2.legend(fontsize=8)
ax2.grid(True)

plt.suptitle('ISIC 2018 — U-Net 학습 곡선 비교', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('/tmp/isic_training_curves.png', dpi=100, bbox_inches='tight')
plt.show()
print('학습 곡선 저장: /tmp/isic_training_curves.png')

# 7-2. 예측 결과 시각화 (최고 모델 사용, 4열: Input / GT / Prob Map / Pred)
best_label = max(all_results, key=lambda k: all_results[k]['best_dice'])
best_model = all_results[best_label]['model']
best_model.eval()
print(f'\n시각화 모델: {best_label}  (Val Dice={all_results[best_label]["best_dice"]:.4f})')

N_VIS = 4  # 시각화 샘플 수
fig, axes = plt.subplots(N_VIS, 4, figsize=(18, N_VIS * 4))

val_tf_vis = A.Compose([A.Resize(IMG_SIZE, IMG_SIZE), A.Normalize(), ToTensorV2()])

for i in range(N_VIS):
    img_path  = val_imgs[i]
    mask_path = val_masks[i]

    # 원본 이미지 + GT
    img_bgr = cv2.imread(img_path)
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    gt_mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    gt_bin  = (gt_mask > 127).astype(np.uint8)

    # 모델 예측
    tensor = val_tf_vis(image=img_rgb)['image'].unsqueeze(0).to(device)
    with torch.no_grad():
        logit = best_model(tensor)                    # (1, 1, H, W)
        prob  = torch.sigmoid(logit).squeeze().cpu().numpy()  # (H, W)

    pred = (prob > 0.5).astype(np.uint8)

    # 원본 크기로 리사이즈
    h0, w0 = gt_bin.shape
    prob_r = cv2.resize(prob, (w0, h0))
    pred_r = (prob_r > 0.5).astype(np.uint8)

    axes[i, 0].imshow(img_rgb)
    axes[i, 0].set_title('Input (Dermoscopy)')
    axes[i, 0].axis('off')

    axes[i, 1].imshow(gt_bin, cmap='gray')
    axes[i, 1].set_title('Ground Truth')
    axes[i, 1].axis('off')

    axes[i, 2].imshow(prob_r, cmap='jet', vmin=0, vmax=1)
    axes[i, 2].set_title('Probability Map')
    axes[i, 2].axis('off')

    axes[i, 3].imshow(pred_r, cmap='gray')
    axes[i, 3].set_title(f'Pred ({best_label.split("(")[0].strip()})')
    axes[i, 3].axis('off')

plt.suptitle(f'ISIC 2018 — 예측 결과 시각화 ({best_label})', fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig('/tmp/isic_prediction_vis.png', dpi=100, bbox_inches='tight')
plt.show()
print('예측 결과 저장: /tmp/isic_prediction_vis.png')

In [ ]:
# ── Cell 8: 최종 정량 평가 + JSON 저장 ───────────────────────────────────────
# rules.md 평가 기준: Dice, Sensitivity, Specificity, AUC (binary seg)

print('\n[전체 모델 종합 평가 — Val Set]')
print(f"{'Loss':<35} {'Dice':>6} {'Sens':>6} {'Spec':>6} {'AUC':>6}")
print('-' * 57)

final_results = {}
for label, v in all_results.items():
    metrics = compute_val_metrics(v['model'], val_loader)
    final_results[label] = {
        'loss_name':     v['loss_name'],
        'alpha':         v['alpha'],
        'best_val_dice': v['best_dice'],
        **metrics,
    }
    print(
        f"{label:<35} "
        f"{metrics['Dice']:>6.4f} "
        f"{metrics['Sens']:>6.4f} "
        f"{metrics['Spec']:>6.4f} "
        f"{metrics['AUC']:>6.4f}"
    )

# ── 바차트 비교 ───────────────────────────────────────────────────────────────
labels_  = list(final_results.keys())
metrics_ = ['Dice', 'Sens', 'Spec', 'AUC']
x = np.arange(len(labels_))

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, metric in zip(axes, metrics_):
    scores = [final_results[lb][metric] for lb in labels_]
    bars   = ax.bar(range(len(labels_)), scores, color=COLORS[:len(labels_)], alpha=0.85)
    ax.set_xticks(range(len(labels_)))
    ax.set_xticklabels(
        [lb.split('(')[0].strip()[:12] for lb in labels_],
        rotation=30, ha='right', fontsize=8
    )
    ax.set_title(metric)
    ax.set_ylim(0, 1.05)
    ax.grid(axis='y', alpha=0.4)
    # 최고값 표시
    best_idx = int(np.argmax(scores))
    bars[best_idx].set_edgecolor('red')
    bars[best_idx].set_linewidth(2.5)
    for bar, score in zip(bars, scores):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{score:.3f}', ha='center', va='bottom', fontsize=7)

plt.suptitle('ISIC 2018 — Loss별 최종 평가 지표 비교', fontsize=13)
plt.tight_layout()
plt.savefig('/tmp/isic_final_metrics.png', dpi=100, bbox_inches='tight')
plt.show()
print('평가 지표 차트 저장: /tmp/isic_final_metrics.png')

# ── JSON 저장 ─────────────────────────────────────────────────────────────────
save_data = {
    'domain':      'ISIC 2018 Skin Lesion Segmentation',
    'model':       'U-Net (ResNet34, ImageNet pretrained)',
    'class_counts': {'BG': class_counts[0], 'FG': class_counts[1]},
    'imbalance_ratio': round(class_counts[0] / class_counts[1], 2),
    'train_size':  len(tr_imgs),
    'val_size':    len(val_imgs),
    'final_epochs': FINAL_EPOCHS,
    'results':     {k: {mk: float(mv) if isinstance(mv, (float, np.floating)) else mv
                        for mk, mv in v.items() if mk != 'model'}
                    for k, v in final_results.items()},
    'best_model':  max(final_results, key=lambda k: final_results[k]['Dice']),
}
with open('/tmp/isic_final_results.json', 'w', encoding='utf-8') as f:
    json.dump(save_data, f, indent=2, ensure_ascii=False)
print('\n최종 결과 저장: /tmp/isic_final_results.json')

print(f"\n최고 모델: {save_data['best_model']}")